# CASMI 2026 — external candidate validation

Builds a compact candidate table from the attached COCONUT CSV-lite file, then measures a database-known holdout. All spectra for each selected structure are excluded from retrieval; `enveda-180` spectra provide timsTOF-like queries. The baseline ranks COCONUT candidates by precursor exact-mass agreement. Run on CPU with internet disabled.


In [ ]:
from pathlib import Path
import subprocess
import sys

INPUT = Path('/kaggle/input')
DATA_DIR = INPUT / 'competitions/enveda-CASMI26-molecule-id-mass-spectra'
if not (DATA_DIR / 'train.parquet').exists():
    raise FileNotFoundError(f'Competition train.parquet not found under {DATA_DIR}')
source_roots = sorted({p.parent for p in INPUT.glob('**/prepare_candidates.py')})
if len(source_roots) != 1:
    raise FileNotFoundError(f'Attach exactly one current CASMI source-code dataset; found {source_roots}')
SRC = source_roots[0]
coconut_files = [p for p in INPUT.glob('**/*') if p.is_file() and 'coconut_csv_lite' in p.name.lower() and p.suffix.lower() in {'.zip', '.csv'}]
if not coconut_files:
    raise FileNotFoundError('Attach the official coconut_csv_lite archive or CSV as a Kaggle Dataset')
COCONUT_FILE = coconut_files[0]
TRAIN = str(DATA_DIR / 'train.parquet')
WORK = Path('/kaggle/working/casmi_database_validation')
WORK.mkdir(parents=True, exist_ok=True)
commit_file = SRC / 'SOURCE_COMMIT.txt'
print('Competition data:', DATA_DIR)
print('Source:', SRC)
print('Source commit:', commit_file.read_text().strip() if commit_file.exists() else 'unknown')
print('COCONUT file:', COCONUT_FILE)

try:
    import rdkit
    parts = tuple(int(p) for p in rdkit.__version__.split('.')[:3])
except (ImportError, ValueError):
    parts = None
if parts != (2026, 3, 3):
    wheels = [p for p in INPUT.glob('**/*.whl') if 'rdkit' in p.name.lower() or '2026.03.3' in str(p)]
    if not wheels:
        raise RuntimeError('Attach the offline RDKit 2026.03.3 wheel dataset')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--upgrade', str(wheels[0])], check=True)
    import rdkit
    parts = tuple(int(p) for p in rdkit.__version__.split('.')[:3])
if parts != (2026, 3, 3):
    raise RuntimeError(f'Expected RDKit 2026.03.3, found {rdkit.__version__}')
print('RDKit:', rdkit.__version__)


In [ ]:
# Create the compact COCONUT table once per notebook session.
CANDIDATES = WORK / 'coconut_candidates.parquet'
subprocess.run([
    sys.executable, str(SRC / 'prepare_candidates.py'),
    '--input', str(COCONUT_FILE), '--output', str(CANDIDATES),
    '--source-version', '2026-09', '--chunk-size', '10000',
], check=True)
print('Candidate table:', CANDIDATES)


In [ ]:
# Hold out only structures present in both train and COCONUT. Keep all query spectra per structure together.
DATABASE_VALIDATION = WORK / 'database_known'
subprocess.run([
    sys.executable, str(SRC / 'prepare_validation.py'),
    '--train', TRAIN, '--output-dir', str(DATABASE_VALIDATION),
    '--mode', 'database', '--candidate-database', str(CANDIDATES),
    '--fraction', '0.003', '--workers', '4', '--seed', '20261001',
    '--query-source', 'enveda-180',
], check=True)


In [ ]:
PREDICTIONS = WORK / 'database_mass_predictions.csv'
subprocess.run([
    sys.executable, str(SRC / 'retrieve_candidates.py'),
    '--candidates', str(CANDIDATES),
    '--queries', str(DATABASE_VALIDATION / 'queries.parquet'),
    '--output', str(PREDICTIONS), '--top-k', '25', '--window-ppm', '1000',
], check=True)
subprocess.run([
    sys.executable, str(SRC / 'evaluate_predictions.py'),
    '--queries', str(DATABASE_VALIDATION / 'queries.parquet'),
    '--predictions', str(PREDICTIONS),
], check=True)


The target structure is guaranteed to exist in the COCONUT table for this validation slice, while its training spectra are excluded. The printed MRR/hit rates therefore measure what exact precursor mass alone can rank within the external candidate pool. A low score points to the need for formula or fragment-based isomer ranking before using GPU time.
